## Clase 10 - Ejercicio: regresión sobre el dataset Swiss

**Rodeyro, Agustín**

Para aprender sobre regresión, vamos a utilizar un dataset clásico llamado Swiss, que proviene originalmente del lenguaje R. Este dataset contiene datos socioeconómicos de 47 provincias suizas a fines del siglo XIX. Cada fila representa una provincia, y las variables reflejan características demográficas y sociales relevantes para ese contexto histórico.

## Variables

- `Location`: Provincia donde se midieron los datos.
- `Fertility`: Tasa de fertilidad (número promedio de hijos por mujer)
- `Agriculture`:` Porcentaje de hombres ocupados en agricultura
- `Examination`: Porcentaje de hombres que completaron exámenes de educación superior
- `Education`: Nivel promedio de educación (escala arbitraria)
- `Catholic`: Porcentaje de población católica
- `Infant.Mortality`: Tasa de mortalidad infantil (por cada 1000 nacidos vivos)

## Que queremos predecir?

Vamos a utilizar este dataset para predecir la tasa de fertilidad en cada provincia mediante diferentes métodos de regresión.

--- 

Siguiendo el procedimiento típico de Machine Learning, vamos a leer los datos y separarlos en los datasets de entrenamiento y testeo utilizando Scikit-Learn...

In [1]:
import pandas as pd

df = pd.read_csv("swiss.csv")

df.head()

,Location,Fertility,Agriculture,Examination,Education,Catholic,Infant.Mortality
0,Courtelary,80.2,17.0,15,12,9.96,22.2
1,Delemont,83.1,45.1,6,9,84.84,22.2
2,Franches-Mnt,92.5,39.7,5,5,93.40,20.2
3,Moutier,85.8,36.5,12,7,33.77,20.3
4,Neuveville,76.9,43.5,17,15,5.16,20.6


In [2]:
print(f"Tenemos {df.shape[0]} observaciones")

Tenemos 47 observaciones


Obtenemos la variable objetivo (`Fertility`) y, por otro lado, los atributos (quitamos `Location` ya que no es un atributo numérico relevante para la regresión)

In [3]:
X = df.drop(["Fertility", "Location"], axis=1)
y = df["Fertility"]

Dado que tenemos pocas observaciones, vamos a separar el dataset en un 50% para entrenamiento y 50% para testeo:

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

## Regresión lineal múltiple

Arranquemos la primera parte del ejercicio. Para eso, vamos a entrenar un modelo de regresión lineal múltiple usando todos los atributos. Para ello debes:

1. Escalar los atributos usando `StandardScaler`
2. Entrenar el modelo usando el dataset de entrenamiento.
3. Obtener las predicciones sobre el dataset de testeo.
4. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

##### COMPLETAR AQUI LO PEDIDO

# 1. Armamos el pipeline: primero escala, después entrena.
#    Poner el escalado adentro del pipeline no es un detalle de prolijidad. Hace que
#    StandardScaler calcule la media y el desvío SOLO con los datos de entrenamiento y
#    después aplique esos mismos valores al set de testeo. Si escaláramos todo junto
#    antes del split, el entrenamiento se estaría enterando de cómo son los datos de
#    testeo.
linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

# 2. Entrenamos con el set de entrenamiento.
linear_pipeline.fit(X_train, y_train)

# 3. Predecimos sobre el set de testeo.
y_pred_linear = linear_pipeline.predict(X_test)

# 4. Calculamos e imprimimos las tres métricas.
mae_linear = mean_absolute_error(y_test, y_pred_linear)
mse_linear = mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)

print("Regresión lineal múltiple")
print(f"  MAE: {mae_linear:.4f}")
print(f"  MSE: {mse_linear:.4f}")
print(f"  R2 : {r2_linear:.4f}")

print("\nCoeficientes:")
for nombre, coef in zip(X_train.columns, linear_pipeline.named_steps["model"].coef_):
    print(f"  {nombre:<18} {coef:8.4f}")

Regresión lineal múltiple
  MAE: 6.0937
  MSE: 64.4572
  R2 : 0.5744

Coeficientes:
  Agriculture         -5.2575
  Examination          3.4068
  Education           -9.9275
  Catholic             7.9742
  Infant.Mortality     3.5841


## Modelo con regularización

Para mejorar nuestro modelo, vamos a explorar técnicas de regresión lineal con regularización, que nos permiten controlar el sobreajuste y seleccionar variables relevantes automáticamente.

Existen dos variantes muy populares:

- Una penaliza la suma de los cuadrados de los coeficientes (regularización L2).
- La otra penaliza la suma del valor absoluto de los coeficientes (regularización L1).

Ambas ayudan a mejorar la generalización, pero una de ellas además puede eliminar variables (coeficientes exactamente cero), lo que ayuda a identificar qué atributos son realmente importantes.

Tu tarea:

1. Elegí correctamente cuál de los dos métodos de regularización usar para este problema. 
    - Pista: Queremos que el modelo sea capaz de hacer una selección automática de variables, dejando fuera aquellas que no aportan.
2. Implementá un pipeline que incluya escalado y el modelo elegido.
3. Buscá automáticamente el mejor valor del hiperparámetro de regularización (alpha) usando validación cruzada usando 3-folds.
4. Entrená el modelo con los datos de entrenamiento y obtené las predicciones para el set de testeo.
5. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.
6. Imprimí los coeficientes resultantes e identificá qué variables fueron eliminadas (coeficiente = 0).

In [6]:
import numpy as np

from sklearn.linear_model import LassoCV, RidgeCV

Paremos un momento para entender qué hacen LassoCV y RidgeCV antes de continuar con la resolución:

> Tanto `LassoCV` como `RidgeCV` son implementaciones de regresión lineal con regularización que incluyen la búsqueda automática del mejor hiperparámetro alpha mediante validación cruzada.
>
> Ambos métodos prueban distintos valores de alpha y eligen el que minimiza el error del modelo, facilitando el proceso de ajuste sin necesidad de una búsqueda manual.
>
> Internamente, utilizan la métrica del error cuadrático medio (MSE) para evaluar el rendimiento del modelo en cada fold de la validación cruzada.
>
> Por ejemplo, si llamás a RidgeCV(alphas=alphas, cv=5), se hará una validación cruzada de 5 folds utilizando los valores de alpha que vos le pases, y se seleccionará el que obtenga el menor MSE promedio.
> 
> Una vez elegido el mejor alpha, el modelo final se entrena con todos los datos de entrenamiento usando ese valor.

¡Listo! Con todo lo que vimos hasta ahora, ya estás en condiciones de resolver esta parte y completar los 6 puntos propuestos

In [7]:
alphas = np.logspace(-4, 1, 500)

##### COMPLETAR AQUI LO PEDIDO

# 1. Elegimos Lasso, o sea regularización L1.
#    La pista pide un modelo que haga selección automática de variables. Lasso penaliza
#    la suma de los valores absolutos de los coeficientes, y eso puede llevarlos a cero
#    exacto, dejando la variable afuera. Ridge penaliza los cuadrados: los achica, pero
#    nunca los anula, así que no sirve para seleccionar variables.
#
# 2. El pipeline incluye el escalado y el modelo elegido.
# 3. LassoCV busca solo el mejor alpha, con validación cruzada de 3 folds sobre el
#    array de alphas que ya venía dado.
lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LassoCV(alphas=alphas, cv=3)),
])

# 4. Entrenamos y predecimos sobre el set de testeo.
lasso_pipeline.fit(X_train, y_train)
y_pred_lasso = lasso_pipeline.predict(X_test)

lasso = lasso_pipeline.named_steps["model"]
print(f"Mejor alpha encontrado por validación cruzada: {lasso.alpha_:.4f}")

# 5. Las mismas tres métricas.
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

print("\nLasso (L1)")
print(f"  MAE: {mae_lasso:.4f}")
print(f"  MSE: {mse_lasso:.4f}")
print(f"  R2 : {r2_lasso:.4f}")

# 6. Coeficientes, marcando cuáles quedaron eliminados.
print("\nCoeficientes:")
eliminadas = []
for nombre, coef in zip(X_train.columns, lasso.coef_):
    if coef == 0:
        eliminadas.append(nombre)
        print(f"  {nombre:<18} {coef:8.4f}   <-- eliminada")
    else:
        print(f"  {nombre:<18} {coef:8.4f}")

print(f"\nVariables eliminadas (coeficiente = 0): {eliminadas}")
print(f"Quedaron {len(X_train.columns) - len(eliminadas)} de "
      f"{len(X_train.columns)} atributos")

Mejor alpha encontrado por validación cruzada: 2.1811

Lasso (L1)
  MAE: 5.9954
  MSE: 64.2943
  R2 : 0.5755

Coeficientes:
  Agriculture         -0.0000   <-- eliminada
  Examination         -1.7624
  Education           -2.4049
  Catholic             1.7196
  Infant.Mortality     3.2004

Variables eliminadas (coeficiente = 0): ['Agriculture']
Quedaron 4 de 5 atributos


## Comparación de modelos y conclusiones

Completá la siguiente tabla con las métricas obtenidas para cada uno de los modelos que entrenaste:

| Modelo                        | MAE    | MSE     | $R^2$  |
| ----------------------------- | ------ | ------- | ------ |
| Regresión Lineal              | 6.0937 | 64.4572 | 0.5744 |
| Lasso (L1)                    | 5.9954 | 64.2943 | 0.5755 |


> ⚠️ Asegurate de cambiar el nombre del modelo `Modelo Regularizado (L1 o L2)` según el modelo que usaste (Lasso o Ridge).

### Justificación

**¿Cuál de los modelos te parece que tuvo un mejor desempeño general?**

Tené en cuenta las tres métricas al responder, y también pensá en la complejidad del modelo (por ejemplo, si eliminó variables innecesarias).

Escribí tu respuesta a continuación:

El mejor desempeño general lo tuvo **Lasso**.

Las tres métricas lo favorecen, aunque por poco: baja el MAE de 6.0937 a 5.9954, baja el MSE de 64.4572 a 64.2943 y sube el $R^2$ de 0.5744 a 0.5755. Las tres apuntan en la misma dirección, así que no es un resultado que dependa de qué métrica se mire.

Dicho eso, con solo 24 observaciones de testeo esas diferencias son demasiado chicas como para decidir por ahí.

Lasso llega a ese resultado usando 4 atributos en vez de 5: eliminó `Agriculture` llevando su coeficiente exactamente a cero. Y tiene sentido que haya caído ese, porque `Agriculture` está fuertemente correlacionada con `Examination` (-0.69) y con `Education` (-0.64), así que la información que aportaba ya estaba en esas dos.

Hay un segundo indicio en los coeficientes. En la regresión lineal sin regularizar aparecen valores grandes y opuestos, como `Education` en -9.93 contra `Catholic` en 7.97. En Lasso el más grande es 3.20. Ese achicamiento importa porque estamos ajustando 5 coeficientes con apenas 23 observaciones de entrenamiento, y según lo que vimos en la teoría los parámetros entrenables tienen que ser muchos menos que las observaciones. Coeficientes tan grandes y de signos opuestos son la señal típica de un modelo que se está apoyando demasiado en el ruido del set de entrenamiento.

En resumen: Lasso iguala o mejora al modelo lineal en las tres métricas, lo hace con una variable menos y con coeficientes mucho más contenidos. Mismo o mejor resultado con un modelo más simple, que es exactamente para lo que sirve la regularización.